# Solutions · 05 · Batch distillation and column efficiency

Worked solutions to the exercises of [notebook 05](../notebooks/05_batch_distillation_and_column_efficiency.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize
from engsep import distillation as dist
from engsep import vle
from engsep.vle import y_from_alpha
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

A 1000 L charge of 40 mol % methanol in water is distilled in a simple still until the distillate averages 70 %. Use the Wilson equilibrium curve. How much is collected, and what is left in the still?

In [2]:
y_mw = lambda x: vle.equilibrium_y(("methanol", "water"), x)
x0 = 0.40
x_end = optimize.brentq(lambda xf: dist.rayleigh(y_mw, x0, xf)["y_avg"] - 0.70, 0.02, 0.39)
r = dist.rayleigh(y_mw, x0, x_end, W0=1000.0)
print(f"distil until the still holds x = {x_end:.3f}: {r['D']:.0f} L collected at an average of 70 % methanol; {r['W']:.0f} L left in the still")

distil until the still holds x = 0.295: 260 L collected at an average of 70 % methanol; 740 L left in the still


With the Wilson curve for methanol-water, a simple still can collect about a third of the charge at 70 %
methanol before the average falls below spec; the remainder is a weak solution for the next batch.

## Exercise 2

For the batch rectification example, compare constant-reflux operation with constant-distillate-composition operation (raise R to hold x_D = 0.9): which collects more distillate above 0.9 from the same charge, and which takes longer (total vapour boiled)?

In [3]:
alpha = 2.4
y_eq = lambda x: y_from_alpha(x, alpha)
cr = dist.batch_constant_reflux(y_eq, x0=0.5, x_final=0.1, R=4.0, N=4)
below = np.nonzero(cr["x_D"] < 0.9)[0]
i_stop = below[0] if below.size else len(cr["x_D"]) - 1
D_const, x_stop_const = cr["D"][i_stop], cr["x_still"][i_stop]
vapour_const = (4.0 + 1) * D_const
# constant distillate composition: at each still composition find the R giving x_D = 0.9 with 4 stages
def R_for(xW):
    def f(R):
        x = y = 0.9
        for _ in range(4):
            x = optimize.brentq(lambda xx: y_eq(xx) - y, 0, 1)
            y = R / (R + 1) * x + 0.9 / (R + 1)
        return x - xW
    return optimize.brentq(f, 0.01, 1e4)
x_limit = optimize.brentq(lambda xW: R_for(xW) - 50.0, 0.22, 0.5)         # the run ends when R reaches a practical limit of 50
xs = np.linspace(0.5, x_limit, 41)
Rs = np.array([R_for(x) for x in xs])
lnW = np.concatenate([[0.0], np.cumsum(np.diff(xs) / (0.9 - xs[1:]))])
W = np.exp(lnW)
D_var = 1 - W
trap = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
vapour_var = trap(Rs + 1, D_var)
print(f"constant reflux R = 4: the distillate falls below 0.9 when the still reaches x = {x_stop_const:.3f}, having collected {D_const:.3f} of the charge and boiled {vapour_const:.2f} charge-volumes of vapour")
print(f"constant composition x_D = 0.9 until R = 50 (still at x = {x_limit:.3f}): {D_var[-1]:.3f} of the charge collected, boiling {vapour_var:.2f} charge-volumes (R from {Rs[0]:.1f} to 50)")
x_tr = 0.9
for _ in range(4):
    x_tr = optimize.brentq(lambda xx: y_eq(xx) - x_tr, 0, 1)                  # total reflux: operating line y = x
print(f"four stages at total reflux cannot take the still below x = {x_tr:.3f} with a 0.9 distillate: the exercise's target of 0.1 is unreachable")

constant reflux R = 4: the distillate falls below 0.9 when the still reaches x = 0.351, having collected 0.258 of the charge and boiled 1.29 charge-volumes of vapour
constant composition x_D = 0.9 until R = 50 (still at x = 0.227): 0.404 of the charge collected, boiling 2.54 charge-volumes (R from 1.5 to 50)
four stages at total reflux cannot take the still below x = 0.213 with a 0.9 distillate: the exercise's target of 0.1 is unreachable


Holding the distillate at specification by raising the reflux collects more on-spec product from the same
charge, but the reflux ratio climbs steeply as the still is depleted and the vapour boiled (the energy and
the time) climbs with it; with only four stages the still cannot be taken below about 0.21 at any reflux,
so the exercise's 0.10 was never reachable. Real batch columns do both: constant reflux for the bulk, rising
reflux for the tail, and a cut point set by the economics.

## Exercise 3

**Implement it yourself:** compute the O'Connell efficiency yourself and the real tray count for the benzene-toluene column of notebook 03 at three viscosities.

In [4]:
alpha_bt = vle.geometric_mean_alpha(("benzene", "toluene"), ideal=True)
N_theo = dist.mccabe_thiele(lambda x: y_from_alpha(x, alpha_bt), 0.95, 0.05, 0.45, 2.0)["stages"] - 1
for mu in (0.2, 0.3, 0.5):
    E = 0.492 * (alpha_bt * mu) ** (-0.245)
    print(f"mu = {mu} mPa s: E = {E:.3f} (library {dist.oconnell_efficiency(alpha_bt, mu):.3f}) -> {int(np.ceil(N_theo / E))} real trays for {N_theo} theoretical")

mu = 0.2 mPa s: E = 0.585 (library 0.585) -> 19 real trays for 11 theoretical
mu = 0.3 mPa s: E = 0.529 (library 0.529) -> 21 real trays for 11 theoretical
mu = 0.5 mPa s: E = 0.467 (library 0.467) -> 24 real trays for 11 theoretical


O'Connell is one line; the real tray count is the theoretical count divided by it and rounded up, and the
uncertainty of the correlation (perhaps 15 %) is a few trays either way - which is why columns get a safety
margin of trays.